# 실습 18. 중단시계열분석과 이중차분법

사회약학 연구방법 노트의 '실습 18 중단시계열분석과 이중차분법'을 따라 하는 노트북입니다. 18장의 예제(65세 이상 외래 환자의 수면진정제 처방 관리 시범사업, 가상 자료)를 시군구 × 월 자료에서 직접 집계해 분절회귀, 대조군이 있는 중단시계열, 이중차분법으로 분석합니다. 셀을 위에서부터 차례로 실행하세요. 자료는 사이트에서 내려받으므로 인터넷 연결이 필요합니다.


## 가. 실습 데이터 준비

시군구 × 월 자료를 읽고, 시간 변수와 처방률을 만들고, 지역을 합친 월별 계열을 만듭니다.

### 셀 1. 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
raw = pd.read_csv(BASE + "sedative_district_month.csv",
                  storage_options=UA)
print(raw.shape)
raw.head(3)

### 셀 2. 자료 구조 확인

In [ ]:
print(raw["unit"].nunique(), "개 시군구,",
      raw["month"].nunique(), "개월")
print(raw["month"].min(), "~", raw["month"].max())

units = raw.groupby("unit").agg(treat=("treat", "first"),
                                months=("month", "size"),
                                mean_pat=("n_pat", "mean"))
print(units["treat"].value_counts())
units.groupby("treat")["mean_pat"].describe().round(0)

### 셀 3. 시간 변수와 처방률 만들기

In [ ]:
d = raw.copy()
d["date"] = pd.to_datetime(d["month"])
d["time"] = ((d["date"].dt.year - 2015) * 12
             + d["date"].dt.month)              # 1, 2, ..., 60
d["post"] = (d["time"] > 36).astype(int)        # 2018년 1월부터 1
d["time_after"] = np.where(d["post"] == 1, d["time"] - 36, 0)
d["rate"] = d["n_sed"] / d["n_pat"] * 1000      # 1,000명당

cols = ["month", "time", "post", "time_after",
        "n_pat", "n_sed", "rate"]
d.loc[d["unit"] == 1, cols].iloc[[0, 35, 36, 37, 59]].round(2)

### 셀 4. 지역을 합친 월별 계열 만들기

In [ ]:
ag = (d.groupby(["treat", "time", "post", "time_after"])
        [["n_sed", "n_pat"]].sum().reset_index())
ag["rate"] = ag["n_sed"] / ag["n_pat"] * 1000

its = ag[ag["treat"] == 1].reset_index(drop=True)    # 시범 지역
comp = ag[ag["treat"] == 0].reset_index(drop=True)   # 비교 지역
print(len(ag), len(its), len(comp))
its.iloc[[0, 1, 35, 36, 37, 59]].round(2)

### 셀 5. 두 계열의 그림

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(its["time"], its["rate"], "o-", ms=3,
        label="Pilot districts")
ax.plot(comp["time"], comp["rate"], "s-", ms=3,
        label="Comparison districts")
ax.axvline(36.5, color="gray", ls="--")           # 시행 시점
ax.set_xticks([1, 13, 25, 37, 49])
ax.set_xticklabels(["2015-01", "2016-01", "2017-01",
                    "2018-01", "2019-01"])
ax.set_xlabel("Month")
ax.set_ylabel("Patients prescribed per 1,000")
ax.legend()
plt.tight_layout()

### 셀 6. 전후 평균만 비교하면

In [ ]:
from scipy import stats

for name, s in [("pilot", its), ("comparison", comp)]:
    pre = s.loc[s["post"] == 0, "rate"]
    po = s.loc[s["post"] == 1, "rate"]
    tt = stats.ttest_ind(po, pre, equal_var=False)
    ci = tt.confidence_interval()
    print(f"{name:<10} before {pre.mean():.2f}  after {po.mean():.2f}"
          f"  diff {po.mean() - pre.mean():.2f}"
          f" ({ci.low:.2f}, {ci.high:.2f})  p = {tt.pvalue:.0e}")

## 나. 중단시계열분석

시범 지역의 월별 계열(its)에 분절회귀를 적합하고, 자기상관을 확인한 뒤 Newey-West 표준오차를 씁니다.

### 셀 7. 분절회귀 (보통의 최소제곱)

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson

f = "rate ~ time + post + time_after"
fit0 = smf.ols(f, data=its).fit()
print(fit0.summary().tables[1])
print("Durbin-Watson:", round(durbin_watson(fit0.resid), 3))

### 셀 8. 잔차의 자기상관 그림

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

r = acf(fit0.resid, nlags=12)
print("lag 1-6 :", np.round(r[1:7], 3))
print("lag 7-12:", np.round(r[7:13], 3))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(its["time"], fit0.resid, "o-", ms=3)
axes[0].axhline(0, color="gray")
axes[0].axvline(36.5, color="gray", ls="--")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Residual")
plot_acf(fit0.resid, lags=12, zero=False, ax=axes[1])
axes[1].set_xlabel("Lag (months)")
plt.tight_layout()

### 셀 9. Newey-West(HAC) 표준오차로 다시 적합

In [ ]:
fit = smf.ols(f, data=its).fit(cov_type="HAC",
                               cov_kwds={"maxlags": 3})
print(fit.summary())

### 셀 10. 시행 12개월째의 효과와 시행 후 기울기

In [ ]:
print(fit.t_test("post + 12 * time_after = 0"))   # 12개월째의 효과
print(fit.t_test("time + time_after = 0"))        # 시행 후 기울기

b = fit.params
cf12 = b["Intercept"] + b["time"] * 48           # 반사실 (time = 48)
eff12 = b["post"] + 12 * b["time_after"]         # 12개월째의 효과
print(f"counterfactual: {cf12:.2f}")
print(f"fitted        : {cf12 + eff12:.2f}")
print(f"relative      : {100 * eff12 / cf12:.1f}%")

### 셀 11. 반사실 선 그리기

In [ ]:
its["fitted"] = fit.predict(its)
its["cf"] = fit.predict(its.assign(post=0, time_after=0))
its["effect"] = its["fitted"] - its["cf"]

pre, po = its[its["post"] == 0], its[its["post"] == 1]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(its["time"], its["rate"], "o", ms=3, color="gray",
        label="Observed")
ax.plot(pre["time"], pre["fitted"], color="C0", label="Fitted")
ax.plot(po["time"], po["fitted"], color="C0")
ax.plot(po["time"], po["cf"], "--", color="C1",
        label="Counterfactual")
ax.axvline(36.5, color="gray", ls="--")
ax.set_xlabel("Month (1 = January 2015)")
ax.set_ylabel("Patients prescribed per 1,000")
ax.legend()
plt.tight_layout()
its.loc[its["time"].isin([37, 48, 60]),
        ["time", "rate", "fitted", "cf", "effect"]].round(2)

### 셀 12. 계절 항을 넣은 모형

In [ ]:
its["sin12"] = np.sin(2 * np.pi * its["time"] / 12)
its["cos12"] = np.cos(2 * np.pi * its["time"] / 12)

fit_s = smf.ols(f + " + sin12 + cos12", data=its).fit(
    cov_type="HAC", cov_kwds={"maxlags": 3})
print(fit_s.summary().tables[1])
print("Durbin-Watson:", round(durbin_watson(fit_s.resid), 3))
print("residual SD  :", round(np.sqrt(fit0.scale), 2), "->",
      round(np.sqrt(fit_s.scale), 2))

### 셀 13. AR(1) 오차 모형과 네 가지 방법의 비교

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(its[["time", "post", "time_after"]])
gls = sm.GLSAR(its["rate"], X, rho=1).iterative_fit(maxiter=50)
print("rho:", np.round(gls.model.rho, 3))

def row(res):
    ci = res.conf_int()
    return [res.params["post"], *ci.loc["post"],
            res.params["time_after"], *ci.loc["time_after"]]

tab = pd.DataFrame(
    {"OLS": row(fit0), "Newey-West (lag 3)": row(fit),
     "AR(1) errors": row(gls), "Seasonal + Newey-West": row(fit_s)},
    index=["level", "lower", "upper", "slope", "lower", "upper"]).T
tab.round(3)

## 다. 대조군이 있는 중단시계열

비교 지역의 계열과 차이의 계열(시범 − 비교)에도 같은 모형을 적합합니다.

### 셀 14. 세 계열에 같은 분절회귀 적합

In [ ]:
diff = its[["time", "post", "time_after"]].copy()
diff["rate"] = its["rate"] - comp["rate"]       # 시범 - 비교

def seg(data):
    return smf.ols(f, data=data).fit(cov_type="HAC",
                                     cov_kwds={"maxlags": 3})

fits = {"pilot": seg(its), "comparison": seg(comp),
        "difference": seg(diff)}
tab3 = pd.DataFrame({k: v.params for k, v in fits.items()})
tab3.loc["effect at 12 months"] = (tab3.loc["post"]
                                   + 12 * tab3.loc["time_after"])
tab3.round(3)

### 셀 15. 차이의 계열 분석 결과

In [ ]:
fit_d = fits["difference"]
print(fit_d.summary().tables[1])
print(fit_d.t_test("post + 12 * time_after = 0"))
print("Durbin-Watson:", round(durbin_watson(fit_d.resid), 2),
      " residual SD:", round(np.sqrt(fit_d.scale), 2))

### 셀 16. 차이의 계열 그림

In [ ]:
diff["fitted"] = fit_d.predict(diff)
diff["cf"] = fit_d.predict(diff.assign(post=0, time_after=0))
pre, po = diff[diff["post"] == 0], diff[diff["post"] == 1]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(diff["time"], diff["rate"], "o", ms=3, color="gray",
        label="Observed difference")
ax.plot(pre["time"], pre["fitted"], color="C0", label="Fitted")
ax.plot(po["time"], po["fitted"], color="C0")
ax.plot(po["time"], po["cf"], "--", color="C1",
        label="Counterfactual")
ax.axvline(36.5, color="gray", ls="--")
ax.set_xlabel("Month (1 = January 2015)")
ax.set_ylabel("Pilot minus comparison, per 1,000")
ax.legend()
plt.tight_layout()

### 셀 17. 교호작용 모형으로 한 번에 적합

In [ ]:
cols = ["treat", "time", "post", "time_after", "rate"]
both = pd.concat([its[cols], comp[cols]], ignore_index=True)
both = both.rename(columns={"treat": "group"})   # 시범 지역 = 1

f_c = "rate ~ (time + post + time_after) * group"
cits = smf.ols(f_c, data=both).fit(
    cov_type="cluster", cov_kwds={"groups": both["time"]})
print(cits.summary().tables[1])
print("ordinary SE:")
print(smf.ols(f_c, data=both).fit().bse[
    ["post:group", "time_after:group"]].round(3))

## 라. 이중차분법

2×2 평균표로 먼저 구하고, 시군구 × 월 자료의 회귀로 같은 값과 군집 강건 표준오차를 구합니다.

### 셀 18. 2×2 평균표와 이중차분 추정값

In [ ]:
m = both.groupby(["group", "post"])["rate"].mean().unstack("post")
m.columns = ["before", "after"]
m["change"] = m["after"] - m["before"]
m.loc["difference"] = m.loc[1] - m.loc[0]
m.round(2)

### 셀 19. 회귀로 구하는 이중차분 추정값 (군집 강건 표준오차)

In [ ]:
panel = d.copy()
panel["w"] = panel.groupby("unit")["n_pat"].transform("mean")

did = smf.wls("rate ~ treat * post", data=panel,
              weights=panel["w"]).fit(
    cov_type="cluster", cov_kwds={"groups": panel["unit"]})
print(did.summary().tables[1])
print("rows:", int(did.nobs), " clusters:", panel["unit"].nunique())

### 셀 20. 시행 전 추세의 차이

In [ ]:
pre = panel[panel["post"] == 0]
pt = smf.wls("rate ~ treat * time", data=pre,
             weights=pre["w"]).fit(
    cov_type="cluster", cov_kwds={"groups": pre["unit"]})
print(pt.summary().tables[1])
lo, hi = pt.conf_int().loc["treat:time"]
print(f"x 30 months: {30 * lo:.2f} ~ {30 * hi:.2f}")

### 셀 21. 사건연구 모형

In [ ]:
panel["half"] = (panel["time"] - 1) // 6 - 6   # -6..-1 전, 0..3 후
terms = []
for h in range(-6, 4):
    if h == -1:                 # 시행 직전 반기 = 기준
        continue
    name = f"lead{-h}" if h < 0 else f"lag{h + 1}"
    panel[name] = ((panel["half"] == h)
                   & (panel["treat"] == 1)).astype(int)
    terms.append(name)

f_es = "rate ~ " + " + ".join(terms) + " + C(unit) + C(time)"
es = smf.wls(f_es, data=panel, weights=panel["w"]).fit(
    cov_type="cluster", cov_kwds={"groups": panel["unit"]})
ev = es.conf_int().loc[terms]
ev.columns = ["lower", "upper"]
ev.insert(0, "coef", es.params[terms])
print(ev.round(2))

### 셀 22. 시행 전 계수의 결합 검정과 사건연구 그림

In [ ]:
leads = ", ".join(f"{t} = 0" for t in terms if "lead" in t)
jt = es.wald_test(leads, scalar=True)        # 시행 전 계수 결합 검정
print(f"joint test of 5 leads: chi2 = {float(jt.statistic):.2f},"
      f" p = {float(jt.pvalue):.2f}")

x = [h for h in range(-6, 4) if h != -1]       # 반기 번호
err = [ev["coef"] - ev["lower"], ev["upper"] - ev["coef"]]

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.errorbar(x, ev["coef"], yerr=err, fmt="o", capsize=3)
ax.plot(-1, 0, "o", color="gray")              # 기준 반기
ax.axhline(0, color="gray")
ax.axvline(-0.5, color="gray", ls="--")
ax.set_xlabel("Half-year relative to the programme start")
ax.set_ylabel("Difference vs. last pre half-year")
plt.tight_layout()

## 과제 정답

**과제 1.** 시범 지역의 계열에서 시행 전 12개월과 시행 후 12개월(time 25–48)만 남겨 나 절의 분절회귀(Newey-West, 시차 3)를 다시 적합하고, 수준 변화와 기울기 변화의 95% 신뢰구간을 60개월 분석과 비교하세요.

### 셀 23. 과제 1 정답. 시행 전후 12개월씩만 쓴 분절회귀

In [ ]:
short = its[(its["time"] > 24) & (its["time"] <= 48)]
fit_12 = smf.ols(f, data=short).fit(cov_type="HAC",
                                    cov_kwds={"maxlags": 3})
print(len(short), "months")
print(fit_12.summary().tables[1])

**과제 2.** 시행 직후 3개월(2018년 1–3월, time 37–39)을 적응 기간으로 보고 빼고 분석하세요. time_after는 그대로 둡니다. 수준 변화, 기울기 변화, 12개월째의 효과를 구하세요.

### 셀 24. 과제 2 정답. 적응 기간 3개월을 뺀 분석

In [ ]:
keep = its[~its["time"].isin([37, 38, 39])]     # 2018년 1-3월 제외
fit_ph = smf.ols(f, data=keep).fit(cov_type="HAC",
                                   cov_kwds={"maxlags": 3})
print(len(keep), "months")
print(fit_ph.summary().tables[1])
print(fit_ph.t_test("post + 12 * time_after = 0"))

**과제 3.** (1) 가중치 없이 이중차분 추정값과 95% 신뢰구간을 구하세요. (2) 시군구마다 '시행 후 평균 − 시행 전 평균'을 하나씩 구해 60개의 변화량으로 두 지역을 비교하세요.

### 셀 25. 과제 3 정답. 가중치 없는 분석과 시군구별 변화량

In [ ]:
# (1) 가중치 없는 이중차분
did_u = smf.ols("rate ~ treat * post", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["unit"]})
lo, hi = did_u.conf_int().loc["treat:post"]
print(f"unweighted DID: {did_u.params['treat:post']:.2f}"
      f" ({lo:.2f}, {hi:.2f})")

# (2) 시군구마다 변화량 하나
chg = (panel.groupby(["unit", "treat", "w", "post"])["rate"]
            .mean().unstack("post").reset_index())
chg["chg"] = chg[1] - chg[0]
print(chg.groupby("treat")["chg"].describe().round(1)[
    ["count", "mean", "std", "min", "max"]])
fit_chg = smf.wls("chg ~ treat", data=chg,
                  weights=chg["w"]).fit(cov_type="HC1")
lo, hi = fit_chg.conf_int().loc["treat"]
print(f"district-level: {fit_chg.params['treat']:.2f}"
      f" ({lo:.2f}, {hi:.2f})  n = {int(fit_chg.nobs)}")